In [66]:
import numpy as np
import matplotlib.pyplot as plt
from mpdaf.obj import Cube
from astropy.coordinates import SkyCoord
import sys
import os
import plotfancy as pf
from matplotlib.patches import Circle, ConnectionPatch, FancyArrowPatch, Arrow
from astropy.visualization import ZScaleInterval

from types import SimpleNamespace
import re
from astropy.io import ascii, fits
from astropy import units as u
from astropy.constants import c as speedoflight
from astropy.table import Table, vstack, hstack, join
from scipy.optimize import curve_fit, root
from astropy.cosmology import Planck18 as cosmo
from astropy import coordinates as coords
from astroquery.sdss import SDSS
from requests.exceptions import ConnectionError
from matplotlib.lines import Line2D
# from hst_phot import *
from prospect.models.templates import TemplateLibrary
from prospect.models import SpecModel
import prospect.fitting as fitting
from prospect.io import write_results as writer
import prospect.io.read_results as reader
from prospect.sources import CSPSpecBasis
from prospect.models import priors
from genesis_metallicity.genesis_metallicity import genesis_metallicity
from mpl_toolkits.axes_grid1.inset_locator import mark_inset, zoomed_inset_axes

# from calculate_jiang19_metallicity import calculate_metallicity_jiang19 as cjm19
sys.path.append('../../../')
import src.ifu_tools.line_ratios as lr
import src.ifu_tools.ifutools as ift

import logging 
logging.getLogger('mpdaf').setLevel(logging.WARNING)

import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning)

In [176]:
lambda_to_spectral_key = {
    'oiii5007_flux': 'OIII-500.7',
    # 'oiii4959_flux': 'OIII-495.9',
    'oii3726_flux':  'OII-372.6',
    'oii3729_flux':  'OII-372.9',

    'halpha_flux':   'H-alpha',
    'hbeta_flux':    'H-beta',
    'hgamma_flux':   'H-gamma',
    # 'hdelta_flux':   'H-delta',
    # 'hepsilon_flux': 'H-epsilon',

    # 'oiii4363_flux': 'OIII-436.3',
    # 'neiii_flux':    'NeIII-386.9',

    # 'nii6583_flux':  'NII-658.3',
    # 'nii6548_flux':  'NII-654.8',

    # 'nev3426_flux':  'NeV-342.6',

    # 'hei5876_flux':  'HeI-587.5',
}

filter_key_to_band = {
    "flux_HST_F435W":       "hst.acs.wfc.F435W",
    "flux_HST_F606W":       "hst.acs.wfc.F606W",
    "flux_HST_F814W":       "hst.acs.wfc.F814W",
    "flux_HST_F125W":       "hst.wfc3.ir.F125W",
    "flux_HST_F160W":       "hst.wfc3.ir.F160W",
    "flux_Spitzer_I1_3.6":  "spitzer.irac.I1",
    "flux_Spitzer_I2_4.5":  "spitzer.irac.I2",
}

redshift = {'z':'redshift', 'object_id':'id'}

master_key = filter_key_to_band | lambda_to_spectral_key | redshift
master_key_inverse = dict((v, k) for k, v in master_key.items())


In [ ]:
test = ascii.read('allsources.csv')
tab = test[test['object_id']!= 'STACK']

#### STATISTICAL FITTING CORRECTION
ratio = tab['oiii5007_flux']/tab['oiii4959_flux']
mask = np.isfinite(ratio)
median = np.median(ratio[mask])
tab['oiii4959_flux'] = tab['oiii4959_flux']*(median/3)

SNR_mask = ((tab['oiii5007_flux']/tab['oiii5007_flux_err'])>5)&((tab['oiii4959_flux']/tab['oiii4959_flux_err'])>5)
EW_mask = (tab['oiii5007_ew']>100)|(tab['hbeta_ew']>50)
selection0 = tab[SNR_mask&EW_mask].copy()

peas = ascii.read('photometry_results.csv')
peas = peas[(peas['flux_Spitzer_I2_4.5']>0)&(peas['flux_Spitzer_I1_3.6']>0)]
common_ids_mask = np.isin(tab['object_id'], peas['object_id'])
selection = tab[common_ids_mask]

peas_final = join(tab, peas, keys='object_id')
peas_final.remove_column('ra_2')
peas_final.remove_column('dec_2')

column_names = ['id','redshift']
column_names.extend(lambda_to_spectral_key.values())
column_names.extend(filter_key_to_band.values())
blank = np.zeros([len(column_names), len(peas_final['object_id'])], dtype=np.float64)*np.nan
cigaletable = Table(blank.T, names=column_names)

for c in column_names:

    if c in list(lambda_to_spectral_key.values()):
        cigaletable[c] = peas_final[master_key_inverse.get(c)]*1e-23*(u.W / u.m**2)
        cigaletable[c+'_err'] = peas_final[master_key_inverse.get(c)+'_err']*1e-23*(u.W / u.m**2)
    elif c in list(filter_key_to_band.values()):
        cigaletable[c] = peas_final[master_key_inverse.get(c)]*1e-3*u.mJy
    else:
        cigaletable[c] = peas_final[master_key_inverse.get(c)]

for c in list(cigaletable.columns)[1:]:
    mask1 = (cigaletable[c]*1e30)<=0.0
    cigaletable[c][mask1] = np.nan

cigaletable.write('cigale/sedpeas.csv', overwrite=True)

props = list(cigaletable.columns[2:])
print('Copy the following into the ini file:')
stri = ''
for p in props:
    stri = stri+p+', '

print(stri)

ValueError: Illegal type <class 'NoneType'> for table item access

In [169]:
peas_final

object_id,ra_1,dec_1,z,angdisp,foreground,cluster_member,lensed,Z_dir,Z_dir_e,Z_j19,Z_j19_e,R23,R23_e,mean_vel_disp,sterr_vel_disp,zcluster,name,oiii5007_flux,oiii5007_flux_err,oiii5007_ew,oiii5007_ew_err,oiii5007_centroid,oiii5007_fwhm,oiii5007_vel_disp,oiii4959_flux,oiii4959_flux_err,oiii4959_ew,oiii4959_ew_err,oiii4959_centroid,oiii4959_fwhm,oiii4959_vel_disp,oii3726_flux,oii3726_flux_err,oii3726_ew,oii3726_ew_err,oii3726_centroid,oii3726_fwhm,oii3726_vel_disp,oii3729_flux,oii3729_flux_err,oii3729_ew,oii3729_ew_err,oii3729_centroid,oii3729_fwhm,oii3729_vel_disp,halpha_flux,halpha_flux_err,halpha_ew,halpha_ew_err,halpha_centroid,halpha_fwhm,halpha_vel_disp,hbeta_flux,hbeta_flux_err,hbeta_ew,hbeta_ew_err,hbeta_centroid,hbeta_fwhm,hbeta_vel_disp,hgamma_flux,hgamma_flux_err,hgamma_ew,hgamma_ew_err,hgamma_centroid,hgamma_fwhm,hgamma_vel_disp,hdelta_flux,hdelta_flux_err,hdelta_ew,hdelta_ew_err,hdelta_centroid,hdelta_fwhm,hdelta_vel_disp,hepsilon_flux,hepsilon_flux_err,hepsilon_ew,hepsilon_ew_err,hepsilon_centroid,hepsilon_fwhm,hepsilon_vel_disp,hzeta_flux,hzeta_flux_err,hzeta_ew,hzeta_ew_err,hzeta_centroid,hzeta_fwhm,hzeta_vel_disp,heta_flux,heta_flux_err,heta_ew,heta_ew_err,heta_centroid,heta_fwhm,heta_vel_disp,oiii4363_flux,oiii4363_flux_err,oiii4363_ew,oiii4363_ew_err,oiii4363_centroid,oiii4363_fwhm,oiii4363_vel_disp,neiii_flux,neiii_flux_err,neiii_ew,neiii_ew_err,neiii_centroid,neiii_fwhm,neiii_vel_disp,nii6583_flux,nii6583_flux_err,nii6583_ew,nii6583_ew_err,nii6583_centroid,nii6583_fwhm,nii6583_vel_disp,nii6548_flux,nii6548_flux_err,nii6548_ew,nii6548_ew_err,nii6548_centroid,nii6548_fwhm,nii6548_vel_disp,sii6716_flux,sii6716_flux_err,sii6716_ew,sii6716_ew_err,sii6716_centroid,sii6716_fwhm,sii6716_vel_disp,sii6731_flux,sii6731_flux_err,sii6731_ew,sii6731_ew_err,sii6731_centroid,sii6731_fwhm,sii6731_vel_disp,nev3426_flux,nev3426_flux_err,nev3426_ew,nev3426_ew_err,nev3426_centroid,nev3426_fwhm,nev3426_vel_disp,fevii3760_flux,fevii3760_flux_err,fevii3760_ew,fevii3760_ew_err,fevii3760_centroid,fevii3760_fwhm,fevii3760_vel_disp,heii4686_flux,heii4686_flux_err,heii4686_ew,heii4686_ew_err,heii4686_centroid,heii4686_fwhm,heii4686_vel_disp,hei5876_flux,hei5876_flux_err,hei5876_ew,hei5876_ew_err,hei5876_centroid,hei5876_fwhm,hei5876_vel_disp,flux_HST_F218W,flux_HST_F225W,flux_HST_F275W,flux_HST_F435W,flux_HST_F606W,flux_HST_F814W,flux_HST_F125W,flux_HST_F160W,flux_Spitzer_I1_3.6,flux_Spitzer_I2_4.5,flux_Spitzer_I4_8.0,flux_Spitzer_M1_24
str29,float64,float64,float64,float64,int64,int64,int64,float64,float64,float64,float64,float64,float64,float64,float64,float64,str12,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64,float64
0d47m35pt895s-6d05m45pt616s,0.7933041666666666,-6.096004444444445,0.4171300625171268,23.503009517081964,0,0,0,8.7694346143